# Neighborhood graph model

This notebook loads the trained message-passing regressor. Training lives in `scripts/train_graph.py` and writes `artifacts/gnn/`.

The graph is an undirected 8-nearest-neighbor graph. A home is linked to another when either lists the other among its 8 nearest neighbors, the haversine distance is at most 5 km, and feature cosine similarity is at least 0. Edge weight is that cosine. Request neighbors are the query's own one-way matches and are labeled `combined`. Messages are divided by the incoming weight sum. The target is natural-log sale price.

ZIP code in the cleaned table is still the raw integer, so the model standardizes it with the training-split mean and standard deviation. The other 17 inputs are the z-scored columns. Map coordinates are the raw latitude and longitude from `kc_house_data.csv`, joined on the row-aligned id.

Held-out metrics below describe this model on its own. They are not a comparison that claims to beat the tabular booster.

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from kingsight.explain.graph import explain_graph
from kingsight.models.graph import FEATURE_ORDER, GraphRegressor

model = GraphRegressor.load(ROOT / "artifacts" / "gnn")
manifest = json.loads((ROOT / "artifacts" / "gnn" / "manifest.json").read_text())
print(model.rule_label)
print(model.description)
print({key: round(value, 4) for key, value in manifest["metrics"].items()})

In [ ]:
index = int(model.test_idx[0])
features = model.features[index].astype(np.float64)
latitude = float(model.latitudes[index])
longitude = float(model.longitudes[index])
property_id = str(model.ids[index])

log_price = model.predict_log_price(features, latitude, longitude, property_id)
print("id", property_id)
print("predicted_log_price", round(log_price, 4), "predicted_usd", round(model.predict_usd(features, latitude, longitude, property_id)))
print("neighbors")
for neighbor in model.neighbors(features, latitude, longitude, property_id, limit=8):
    print(f"  {neighbor['id']}  {neighbor['distance_meters']:.0f} m  {neighbor['edge_type']}")

edited = features.copy()
edited[FEATURE_ORDER.index("sqft_living")] += 2.0
print("edited living-area log price", round(model.predict_log_price(edited, latitude, longitude, property_id), 4))
explanation = explain_graph(model, features, latitude, longitude, property_id)
print(explanation["method"])
print("reference", round(explanation["reference_output"], 4))
for item in explanation["features"][:6]:
    print(f"  {item['label']}: {item['contribution']:+.4f}")